# B / 历史特征生成与验证（B_v2 · 口径与关联核验更新2026-10-11）

将特征变换、逐年写出和独立检查展开为可编辑代码。现有全量结果共有 48,538,627 行。

v2 新增年度描述、频率与非零强度图；历史特征扩展至 26 个数值输入。年度固定站网交集为空，不能据此声称全国年度趋势。连续湿槽和距上次降雨时间仅观察过去 60 分钟，并带截断标记。station 编码仍由 C 仅在训练集拟合。随机核验使用固定种子，保留跨年检查。

默认开关均为 `False`，因此 **Run All 只查看已有结果，不启动全量 Spark、不覆盖原结果**。

- 阅读代码：代码单元格内可以直接修改，不只是 `%run` 调用脚本。
- 运行一个单元格：`Shift + Enter`。按从上到下的顺序执行；先安装依赖，再设置参数。
- 修改后重跑：打开对应开关，重新运行参数格，然后运行该部分的全部代码格。
- 全量计算需要使用项目 `.venv` 的 Python 3.11，并在启动 Jupyter 前设置 Java 17。
- 结果写入带时间戳的新目录；原八年正式结果继续保留。
- 默认导入的 `b_common` 是共享的路径、Spark 和文件保存工具；分析和特征变换主体均在本 Notebook 展开。


## 1. 路径与共享工具

In [1]:
from pathlib import Path
import sys, json, csv, time, calendar, collections, math, shutil
from datetime import datetime
from decimal import Decimal
from IPython.display import display, HTML, Image

# 如果组员在另一台电脑上运行，只需修改这里的项目根目录。
PROJECT_ROOT = Path('/Users/zhu/Downloads/5208/task2/DSA5208_Project2').resolve()

# Jupyter 通常以 Notebook 所在目录为工作目录；也支持从项目根目录启动。
candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
BROOT = next((p for p in candidates if (p / 'src/b_common.py').is_file()), None)
if BROOT is None:
    BROOT = PROJECT_ROOT / 'B_delivery'
assert (BROOT / 'src/b_common.py').is_file(), '先把 B_source_report.zip 解压到项目根目录'
assert (PROJECT_ROOT / 'scripts/data_release.py').is_file(), '请检查 PROJECT_ROOT'
sys.path.insert(0, str(BROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from b_common import csv_write, dump, log, metadata, release, rows, start
root = PROJECT_ROOT
d = release(root)

# 直接显示汇总 CSV，无需安装 pandas。
def show_csv(path, limit=15):
    data = rows(path)
    if not data:
        print('空表：', path)
        return
    from html import escape
    names = list(data[0])
    header = ''.join('<th>'+escape(k)+'</th>' for k in names)
    body = ''.join('<tr>'+''.join('<td>'+escape(str(r[k]))+'</td>' for k in names)+'</tr>' for r in data[:limit])
    display(HTML('<div style="overflow-x:auto"><table><tr>'+header+'</tr>'+body+'</table></div>'))
    print(f'显示 {min(limit,len(data))} / {len(data)} 行；来源：{path}')

print('项目：', PROJECT_ROOT)
print('B 包：', BROOT)
print('Python：', sys.version.split()[0])


项目： /Users/zhu/Downloads/5208/task2/DSA5208_Project2
B 包： /Users/zhu/Downloads/5208/task2/DSA5208_Project2/B_delivery
Python： 3.11.9


## 2. 查看已有验证结果

In [2]:
EXISTING_FEATURES = BROOT / 'results/features_v2'
show_csv(EXISTING_FEATURES / 'yearly_validation.csv')
print((EXISTING_FEATURES / 'validation.json').read_text())
print((EXISTING_FEATURES / 'inference_adapter_validation.json').read_text())


year,rows,history_complete,invalid_15m,invalid_30m,invalid_60m,invalid_previous_15m,invalid_model_nulls_complete,invalid_recency_semantics,status
2017,5256020,4941980,0,0,0,0,0,0,PASS
2018,4594246,4028323,0,0,0,0,0,0,PASS
2019,5026862,4965109,0,0,0,0,0,0,PASS
2020,6334473,6264593,0,0,0,0,0,0,PASS
2021,7044646,6995148,0,0,0,0,0,0,PASS
2022,7133230,7091648,0,0,0,0,0,0,PASS
2023,6742890,6704161,0,0,0,0,0,0,PASS
2024,6406260,6356871,0,0,0,0,0,0,PASS


显示 8 / 8 行；来源：/Users/zhu/Downloads/5208/task2/DSA5208_Project2/B_delivery/results/features_v2/yearly_validation.csv
{
  "status": "PASS",
  "rows": 48538627,
  "history_complete": 47347833,
  "independent_probes": 513,
  "yearly_counts_match_A_targets": true,
  "window_null_semantics": true,
  "no_future_columns": true,
  "key_uniqueness": "preserved from verified observations; windows neither duplicate nor group anchors",
  "input_integrity": {
    "status": "PASS",
    "scope": "full",
    "checked_files": 549,
    "release_id": "rainfall_v1_r3",
    "data_version": "v1",
    "config_hash": "e5258b99925606c8ba88207cb44a6756514fa84cd60f37adea3f9db64103248b",
    "release_manifest_sha256": "8ebfb9ff0a91cb7e171fadee1d4c25069a74cc779588671ace1b4cf247134b3e"
  },
  "feature_set_version": "B_v2",
  "complete_history_model_inputs_nonnull": true,
  "recency_censor_semantics": true
}

{
  "status": "PASS",
  "station_id": "S24",
  "prediction_epoch_s": 1557901800,
  "compared_features": [
   

## 3. 运行参数

默认不生成新特征。修改后需要重跑时设 `RECOMPUTE_FEATURES=True`。默认仍计算全部八年；需额外约 1 GB 输出空间和 Spark 临时空间。

In [3]:
RECOMPUTE_FEATURES = False
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
NEW_FEATURES = BROOT / 'results' / ('features_notebook_' + RUN_TAG)
spark = None
print('新特征目录：', NEW_FEATURES)


新特征目录： /Users/zhu/Downloads/5208/task2/DSA5208_Project2/B_delivery/results/features_notebook_20261011_113827


## 4. 时间工具

In [4]:
from datetime import datetime, timedelta, timezone
SGT = timezone(timedelta(hours=8))
def epoch(year,month=1,day=1,hour=0,minute=0):
    return int(datetime(year,month,day,hour,minute,tzinfo=SGT).timestamp())


## 5. 可编辑的特征变换函数

这是真正参与后续计算的函数，可以直接改窗口、增加特征。`rangeBetween` 使用秒数；必须在计算完窗口后才筛选锚点年份。修改后先重新运行此格。

In [5]:
def make_features(obs, year):
    """Causal features. Run length and recency are bounded by a complete 60m history."""
    from pyspark.sql import functions as F, Window
    begin, end = epoch(year), epoch(year + 1)
    x = (obs.filter(F.col('year').isin(year - 1, year))
        .filter(F.col('slot_end_epoch_s').between(begin - 3300, end - 1))
        .select('station_id', F.col('slot_end_epoch_s').alias('prediction_epoch_s'),
            F.col('slot_end_ts').alias('prediction_ts'), F.col('rain_mm').alias('current_5m_mm')))
    base = Window.partitionBy('station_id').orderBy('prediction_epoch_s')
    for label, left, right, count in [
        ('15m', -600, 0, 3), ('30m', -1500, 0, 6),
        ('60m', -3300, 0, 12), ('previous_15m', -1500, -900, 3)
    ]:
        w = base.rangeBetween(left, right)
        x = x.withColumn('n_valid_' + label, F.count('current_5m_mm').over(w))
        x = x.withColumn('rain_sum_' + label + '_mm',
            F.when(F.col('n_valid_' + label) == count, F.sum('current_5m_mm').over(w)))
    w60 = base.rangeBetween(-3300, 0)
    complete = F.col('n_valid_60m') == 12
    x = (x.withColumn('history_complete_60m', complete)
        .withColumn('rain_max_60m_mm', F.when(complete, F.max('current_5m_mm').over(w60)))
        .withColumn('rain_std_60m_mm', F.when(complete, F.stddev_pop('current_5m_mm').over(w60)))
        .withColumn('wet_slots_60m', F.when(complete,
            F.sum((F.col('current_5m_mm') > 0).cast('int')).over(w60)))
        .withColumn('rain_change_15m_mm', F.col('rain_sum_15m_mm') - F.col('rain_sum_previous_15m_mm')))
    # Exact offsets, not preceding-row positions: gaps remain unknown.
    for minutes in [5, 10, 15, 30]:
        w = base.rangeBetween(-60 * minutes, -60 * minutes)
        x = x.withColumn(f'lag_{minutes}m_mm',
            F.when(F.count('current_5m_mm').over(w) == 1, F.sum('current_5m_mm').over(w)))
    x = x.withColumn('wet_current', (F.col('current_5m_mm') > 0).cast('int'))
    for minutes in [5, 10]:
        x = x.withColumn(f'wet_lag_{minutes}m', (F.col(f'lag_{minutes}m_mm') > 0).cast('int'))
    x = (x.withColumn('rain_change_5m_mm', F.col('current_5m_mm') - F.col('lag_5m_mm'))
        .withColumn('rain_ratio_15m_30m',
            F.when(F.col('n_valid_30m') == 6,
                F.when(F.col('rain_sum_30m_mm') > 0,
                    F.col('rain_sum_15m_mm') / F.col('rain_sum_30m_mm')).otherwise(F.lit(0.0)))))
    x = (x.withColumn('_last_dry_epoch', F.max(F.when(F.col('current_5m_mm') == 0,
            F.col('prediction_epoch_s'))).over(w60))
        .withColumn('_last_wet_epoch', F.max(F.when(F.col('current_5m_mm') > 0,
            F.col('prediction_epoch_s'))).over(w60)))
    x = (x.withColumn('consecutive_wet_slots_60m', F.when(complete,
            F.when(F.col('_last_dry_epoch').isNull(), F.lit(12))
             .otherwise(((F.col('prediction_epoch_s') - F.col('_last_dry_epoch')) / 300).cast('int'))))
        .withColumn('wet_run_left_censored_60m', F.when(complete,
            F.col('_last_dry_epoch').isNull().cast('int')))
        .withColumn('no_rain_observed_60m', F.when(complete,
            F.col('_last_wet_epoch').isNull().cast('int')))
        # Raw recency remains null when no wet slot is observed; the exact age is unknown.
        .withColumn('minutes_since_last_rain_60m', F.when(complete & F.col('_last_wet_epoch').isNotNull(),
            (F.col('prediction_epoch_s') - F.col('_last_wet_epoch')) / 60.0))
        # 60 is a documented censoring code, accompanied by no_rain_observed_60m.
        # It does not assert that the last rainfall ended exactly 60 minutes ago.
        .withColumn('minutes_since_last_rain_60m_capped', F.when(complete,
            F.coalesce(F.col('minutes_since_last_rain_60m'), F.lit(60.0))))
        .drop('_last_dry_epoch', '_last_wet_epoch'))
    clock = F.hour('prediction_ts') + F.minute('prediction_ts') / F.lit(60.0)
    for name, angle in [('hour', 2 * math.pi * clock / 24.0),
                        ('month', 2 * math.pi * (F.month('prediction_ts') - 1) / 12.0)]:
        x = x.withColumn(name + '_sin', F.sin(angle)).withColumn(name + '_cos', F.cos(angle))
    return x.filter(F.col('prediction_epoch_s').between(begin, end - 1))


## 6. 独立时间戳核验函数

该函数用 Python 字典重新查时间戳，并非再次调用同一套 Spark 窗口逻辑。修改特征后，相应补充验证，不能只更新列名。

In [6]:
def validate_probes(spark, obs, feature_root, out):
    """Independent Python lookup for boundary cases + seeded 20-station/400-anchor probes."""
    import hashlib, random
    from collections import defaultdict
    from pyspark.sql import functions as F
    seed = 20261011
    boundary_stations = ['S24', 'S77', 'S113']
    anchors = [epoch(y, m, d, h, mi) for y in range(2017, 2025)
        for m, d, h, mi in [(1,1,0,0),(1,1,0,30),(5,15,14,30),(9,30,23,55),(12,31,23,55)]]
    boundary_keys = {(s, t) for s in boundary_stations for t in anchors}
    station_ids = sorted(r['station_id'] for r in obs.groupBy('station_id').count().filter(F.col('count') >= 20).select('station_id').collect())
    stations = random.Random(seed).sample(station_ids, min(20, len(station_ids)))
    sampled = (obs.filter(F.col('station_id').isin(stations))
        .select('station_id', 'slot_end_epoch_s')
        .filter(F.pmod(F.xxhash64('station_id', 'slot_end_epoch_s', F.lit(seed)), F.lit(1000)) == 0)
        .collect())
    per_station = defaultdict(list)
    for r in sampled:
        per_station[r['station_id']].append(r['slot_end_epoch_s'])
    random_keys = set()
    for s in stations:
        times = sorted(per_station[s], key=lambda t: hashlib.sha256(f'{seed}|{s}|{t}'.encode()).digest())
        # Sparse stations may need an exact hash-ordered fallback to supply 20 anchors.
        if len(times) < 20:
            times = [r['slot_end_epoch_s'] for r in obs.filter(F.col('station_id') == s)
                .select('slot_end_epoch_s').orderBy(F.xxhash64('slot_end_epoch_s', F.lit(seed))).limit(20).collect()]
        random_keys.update((s, t) for t in times[:20])
    keys = sorted(boundary_keys | random_keys)
    key_frame = spark.createDataFrame(keys, 'station_id string, prediction_epoch_s long')
    needed = sorted({(s, t + offset) for s, t in keys for offset in range(-3300, 1, 300)})
    need_frame = spark.createDataFrame(needed, 'station_id string, slot_end_epoch_s long')
    source = obs.select('station_id', 'slot_end_epoch_s', 'rain_mm').join(
        F.broadcast(need_frame), ['station_id', 'slot_end_epoch_s'], 'inner')
    lookup = {(r['station_id'], r['slot_end_epoch_s']): r['rain_mm'] for r in source.collect()}
    features = spark.read.parquet(str(feature_root))
    probes = features.join(F.broadcast(key_frame), ['station_id', 'prediction_epoch_s'], 'inner').collect()
    assert len(random_keys) == 400, 'Expected 20 eligible stations with 20 distinct anchors each'
    assert len(probes) > 400, 'Expected boundary probes plus 400 random probes'
    def equal(actual, expected, label):
        assert (actual is None) == (expected is None), (label, actual, expected)
        if expected is not None:
            assert math.isclose(actual, expected, rel_tol=1e-10, abs_tol=1e-8), (label, actual, expected)
    checked = []
    for row in probes:
        r = row.asDict(); s, t = r['station_id'], r['prediction_epoch_s']
        for label, offsets in [('15m', range(-600,1,300)), ('30m', range(-1500,1,300)),
                               ('60m', range(-3300,1,300)), ('previous_15m', range(-1500,-899,300))]:
            values = [lookup.get((s, t + offset)) for offset in offsets]
            known = [v for v in values if v is not None]
            assert r['n_valid_' + label] == len(known), (s, t, label)
            equal(r['rain_sum_' + label + '_mm'], sum(known) if len(known) == len(values) else None, label)
        values = [lookup.get((s, t + offset)) for offset in range(-3300,1,300)]
        complete = all(v is not None for v in values)
        assert r['history_complete_60m'] == complete
        for minutes in [5,10,15,30]:
            equal(r[f'lag_{minutes}m_mm'], lookup.get((s,t-minutes*60)), f'lag{minutes}')
        equal(r['wet_current'], int(lookup[(s,t)] > 0), 'wet_current')
        for minutes in [5,10]:
            v = lookup.get((s,t-minutes*60))
            equal(r[f'wet_lag_{minutes}m'], int(v > 0) if v is not None else None, 'wet_lag')
        lag = lookup.get((s,t-300))
        equal(r['rain_change_5m_mm'], lookup[(s,t)] - lag if lag is not None else None, 'change5')
        last30 = [lookup.get((s,t+o)) for o in range(-1500,1,300)]
        ratio = None
        if all(v is not None for v in last30):
            ratio = sum(last30[-3:])/sum(last30) if sum(last30) > 0 else 0.0
        equal(r['rain_ratio_15m_30m'], ratio, 'ratio')
        if complete:
            mean = sum(values)/12
            equal(r['rain_max_60m_mm'], max(values), 'max')
            equal(r['rain_std_60m_mm'], math.sqrt(sum((v-mean)**2 for v in values)/12), 'std')
            equal(r['wet_slots_60m'], sum(v > 0 for v in values), 'wet count')
            run = 0
            for v in reversed(values):
                if v <= 0: break
                run += 1
            ages = [i*5 for i,v in enumerate(reversed(values)) if v > 0]
            age = min(ages) if ages else None
            equal(r['consecutive_wet_slots_60m'], run, 'run')
            equal(r['wet_run_left_censored_60m'], int(run == 12), 'left censor')
            equal(r['no_rain_observed_60m'], int(not ages), 'no rain')
            equal(r['minutes_since_last_rain_60m'], age, 'raw age')
            equal(r['minutes_since_last_rain_60m_capped'], age if age is not None else 60, 'capped age')
        else:
            for column in ['rain_max_60m_mm','rain_std_60m_mm','wet_slots_60m','consecutive_wet_slots_60m',
                           'wet_run_left_censored_60m','no_rain_observed_60m',
                           'minutes_since_last_rain_60m','minutes_since_last_rain_60m_capped']:
                equal(r[column], None, column)
        checked.append({'station_id':s, 'prediction_epoch_s':t,
            'history_complete_60m':complete,
            'probe_type':'boundary' if (s,t) in boundary_keys else 'seeded_random', 'status':'PASS'})
    csv_write(out/'independent_timestamp_probes.csv', sorted(checked,key=lambda r:(r['station_id'],r['prediction_epoch_s'])))
    dump(out/'probe_design.json', {'seed':seed, 'random_stations':stations,
        'random_anchor_count':len(random_keys), 'boundary_keys_requested':len(boundary_keys),
        'actual_checked':len(probes), 'selection':'fixed seeded stations; stable hash sample and per-station hash order'})
    return len(probes)


## 7. 启动 Spark 和正式数据完整性校验

In [7]:
if RECOMPUTE_FEATURES:
    assert sys.version_info[:2] == (3,11), '请使用项目 .venv 的 Python 3.11 内核'
    begin = time.monotonic()
    out = NEW_FEATURES
    out.mkdir(parents=True, exist_ok=False)
    verification = d.verify_release(root, 'full')
    spark = start(root, 'B notebook historical features')
else:
    print("已跳过。需要重跑时，在参数单元格把 RECOMPUTE_FEATURES 改成 True，再从该部分第一格按顺序运行。")


已跳过。需要重跑时，在参数单元格把 RECOMPUTE_FEATURES 改成 True，再从该部分第一格按顺序运行。


## 8. 读取源表和逐年参照数量

In [8]:
if RECOMPUTE_FEATURES:
    from pyspark.sql import functions as F
    tables=d.read_tables(spark,root,'full',verify=False)
    obs=tables['observations']
    log('Reading reference yearly row counts and history-completeness counts')
    references={r['year']:r.asDict() for r in tables['targets'].groupBy('year').agg(
        F.count('*').alias('rows'),F.sum(F.col('historical_valid').cast('long')).alias('history_complete')).collect()}
    feature_root=out/'parquet'
    summaries=[]
else:
    print("已跳过。需要重跑时，在参数单元格把 RECOMPUTE_FEATURES 改成 True，再从该部分第一格按顺序运行。")


已跳过。需要重跑时，在参数单元格把 RECOMPUTE_FEATURES 改成 True，再从该部分第一格按顺序运行。


## 9. 逐年生成、保存和检查特征

此单元格会写出全部八年。计算期间不要同时启动另一个全量 Spark Notebook。

In [9]:
if RECOMPUTE_FEATURES:
    for year in range(2017,2025):
        log(f'{year}: computing time windows with previous-year context')
        f=make_features(obs,year)
        target=feature_root/f'year={year}'
        f.write.mode('errorifexists').parquet(str(target))
        log(f'{year}: checking saved feature counts and null semantics')
        saved=spark.read.parquet(str(target))
        expressions=[F.count('*').alias('rows'),F.sum(F.col('history_complete_60m').cast('long')).alias('history_complete')]
        for label,n in [('15m',3),('30m',6),('60m',12),('previous_15m',3)]:
            invalid=((F.col('n_valid_'+label)==n)!=F.col('rain_sum_'+label+'_mm').isNotNull())
            expressions.append(F.sum(invalid.cast('long')).alias('invalid_'+label))
        from B_feature_api import NUMERIC_FEATURES
        from functools import reduce
        any_model_null = reduce(lambda left,right:left|right,
            (F.col(c).isNull() for c in NUMERIC_FEATURES))
        expressions.append(F.sum((F.col('history_complete_60m') & any_model_null).cast('long')).alias('invalid_model_nulls_complete'))
        invalid_age = F.col('history_complete_60m') & (
            F.col('minutes_since_last_rain_60m').isNull() != (F.col('no_rain_observed_60m') == 1))
        expressions.append(F.sum(invalid_age.cast('long')).alias('invalid_recency_semantics'))
        stats=saved.agg(*expressions).first().asDict()
        assert stats['invalid_model_nulls_complete']==0 and stats['invalid_recency_semantics']==0, stats
        assert stats['rows']==references[year]['rows'],(year,stats,references[year])
        assert stats['history_complete']==references[year]['history_complete'],(year,stats,references[year])
        assert all(stats['invalid_'+label]==0 for label in ['15m','30m','60m','previous_15m']),stats
        summaries.append({'year':year,**stats,'status':'PASS'})
        csv_write(out/'yearly_validation.csv',summaries)
        dump(out/'feature_schema.json',saved.schema.jsonValue())
        log(f'{year}: PASS, {stats["rows"]:,} anchors')
else:
    print("已跳过。需要重跑时，在参数单元格把 RECOMPUTE_FEATURES 改成 True，再从该部分第一格按顺序运行。")


已跳过。需要重跑时，在参数单元格把 RECOMPUTE_FEATURES 改成 True，再从该部分第一格按顺序运行。


## 10. 独立验证、禁止未来字段检查和结果记录

In [10]:
if RECOMPUTE_FEATURES:
    log('Validating independent timestamp probes, including year boundaries')
    n_probes=validate_probes(spark,obs,feature_root,out)
    columns=spark.read.parquet(str(feature_root)).columns
    prohibited=['future_30m_mm','target_valid','model_anchor_eligible','fold1_role','fold2_role','fold3_role','final_role','year_boundary_safe']
    assert not set(columns).intersection(prohibited)
    dump(out/'validation.json',{'status':'PASS','rows':sum(r['rows'] for r in summaries),
        'history_complete':sum(r['history_complete'] for r in summaries),'independent_probes':n_probes,
        'yearly_counts_match_A_targets':True,'window_null_semantics':True,'no_future_columns':True,
        'key_uniqueness':'preserved from verified observations; windows neither duplicate nor group anchors',
        'input_integrity':verification,'feature_set_version':'B_v2',
        'complete_history_model_inputs_nonnull':True,'recency_censor_semantics':True})
    metadata(root,out,'python src/B_build_features.py --root PROJECT_ROOT',time.monotonic()-begin, {'feature_set_version':'B_v2'})
    log('All historical features PASS')
else:
    print("已跳过。需要重跑时，在参数单元格把 RECOMPUTE_FEATURES 改成 True，再从该部分第一格按顺序运行。")


已跳过。需要重跑时，在参数单元格把 RECOMPUTE_FEATURES 改成 True，再从该部分第一格按顺序运行。


## 11. 关闭 Spark

In [11]:
if spark is not None:
    spark.stop()
    spark = None
    print('Spark 已关闭')


## 12. 保存这次修改后的特征函数

特征生成和 C 的预测入口必须使用相同变换。下面会把当前 Notebook 中的函数导出到**新结果目录**，不会覆盖原脚本。交接时，把导出的脚本连同新特征交给 C；C 需使用该版本同步预测入口。

In [12]:
if RECOMPUTE_FEATURES:
    import inspect
    original = (BROOT / 'src/B_build_features.py').read_text()
    import ast, textwrap
    for name, function in [('make_features', make_features), ('validate_probes', validate_probes)]:
        tree = ast.parse(original)
        node = next(n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name==name)
        lines = original.splitlines(keepends=True)
        lines[node.lineno-1:node.end_lineno] = [textwrap.dedent(inspect.getsource(function)).rstrip()+'\n']
        original = ''.join(lines)
    ast.parse(original)
    (NEW_FEATURES / 'B_build_features_notebook_version.py').write_text(original)
    print('当前函数已导出：', NEW_FEATURES / 'B_build_features_notebook_version.py')
else:
    print("已跳过。需要重跑时，在参数单元格把 RECOMPUTE_FEATURES 改成 True，再从该部分第一格按顺序运行。")


已跳过。需要重跑时，在参数单元格把 RECOMPUTE_FEATURES 改成 True，再从该部分第一格按顺序运行。


## 13. 记录修改说明

In [13]:
if RECOMPUTE_FEATURES:
    experiment = json.loads((BROOT / 'configs/B_experiment.json').read_text())
    experiment['notebook'] = 'B_Historical_Features.ipynb'
    experiment['changes'] = '在这里准确填写窗口、字段等修改；同时更新 features 配置'
    dump(NEW_FEATURES / 'notebook_experiment.json', experiment)
    meta = json.loads((NEW_FEATURES / 'result_metadata.json').read_text())
    meta['experiment_config_file'] = 'notebook_experiment.json'
    meta['experiment_config_hash'] = d.sha256(NEW_FEATURES / 'notebook_experiment.json')
    meta['notebook_exported_code_hash'] = d.sha256(NEW_FEATURES / 'B_build_features_notebook_version.py')
    meta['outputs'] = [{'path':p.relative_to(NEW_FEATURES).as_posix(), 'bytes':p.stat().st_size, 'sha256':d.sha256(p)}
                       for p in sorted(NEW_FEATURES.rglob('*')) if p.is_file() and p.name!='result_metadata.json' and not p.name.startswith('.')]
    dump(NEW_FEATURES / 'result_metadata.json', meta)
    print('新特征及修改记录：', NEW_FEATURES)
else:
    print("已跳过。需要重跑时，在参数单元格把 RECOMPUTE_FEATURES 改成 True，再从该部分第一格按顺序运行。")


已跳过。需要重跑时，在参数单元格把 RECOMPUTE_FEATURES 改成 True，再从该部分第一格按顺序运行。


## C 的交接边界

- 主键是 `station_id + prediction_epoch_s`，一对一关联标签。
- 当前及历史雨量只用 t 及之前的观测；标签和未来资格字段不可进入模型特征。
- 窗口不完整时保留 null，不能把缺测补成零。
- 本 Notebook 不做阈值展开、学习编码或训练模型；这些由 C 在各轮训练集内完成。
- 修改特征后需更新 `NUMERIC_FEATURES` 和预测接口，重新验证离线与在线计算一致，不能直接沿用旧版本的验证结论。
- 2024 标签继续保留为最终测试，不用于选择特征或模型参数。

## 补充：时间口径与证据边界

以下说明依据A冻结配置与B代码；区间含义是项目假设，C最终实现及三方交接尚待确认。

# 预测时间与雨量区间：A/B/C 交接口径

## 1. 项目当前约定与证据边界

A 的冻结 `configs/project.json` 选择 L1/D1：雨量解释为截至对齐后 `slot_end_ts` 的五分钟区间量。`event_ts` 是保留的原始解析时间；仅2017年且 epoch 秒除300余299的记录向后对齐1秒，其余不变。原始时间和对齐标记仍可追溯。该区间解释是项目明确假设，不应写成已由官方文档独立证明。

预测时刻 `t = prediction_epoch_s = slot_end_epoch_s`，时区 Asia/Singapore。Task 1 按 `rain_period_start_ts = slot_end_ts − 300秒` 归属年月日与小时。例如末端00:00的五分钟雨量归前一天23:55起始区间。

## 2. t 时刻可用的最后一条观测

回溯事件时间假设下，末端为 t、对应区间 `(t−5min,t]` 的观测 R(t) 已完成且可用，可作为当前槽输入；所有输入末端不得大于 t。最近60分钟使用12个精确末端 `t−55min,…,t−5min,t`，对应区间并集 `(t−60min,t]`。

这一假设不等于真实在线可用性已经验证：数据可能有发布或更新延迟，当前方案没有按 `update_ts/reading_update_ts` 重建 as-of 数据版本。若真实时刻t尚未获得R(t)，必须重新定义可用观测截止点并同步特征与评价；不能直接宣称当前结果是实时部署表现。

## 3. 未来30分钟标签：只包含下面六个区间

| 槽 | 末端 | 雨量区间 |
|---|---|---|
| 1 | t+5min | (t,t+5min] |
| 2 | t+10min | (t+5min,t+10min] |
| 3 | t+15min | (t+10min,t+15min] |
| 4 | t+20min | (t+15min,t+20min] |
| 5 | t+25min | (t+20min,t+25min] |
| 6 | t+30min | (t+25min,t+30min] |

`future_30m_mm = R(t+300)+R(t+600)+…+R(t+1800)`，区间并集为 `(t,t+30min]`，**不包含当前 R(t)**。六槽全部有效才有累计量，否则 null。阈值标签为 `future_30m_mm > threshold_mm`；相等为负例，未知不能当负例。累计及严格阈值比较沿用A的Decimal语义。

例如 t=2019-05-15 14:30 SGT：最后可用观测对应(14:25,14:30]；标签依次对应(14:30,14:35]至(14:55,15:00]。历史和未来用精确时间位置，不能使用“前12行/后6行”替代；跨年必须读取必要上下文。

## 4. 确认状态

| 角色 | 当前可核实状态 | 仍需确认 |
|---|---|---|
| A | 冻结配置及标签实现采用上述末端/六槽约定 | 提交前再次核对原始时间语义的项目假设与延迟限制 |
| B | 特征与预测接口采用 t 及以前观测；跨年/缺槽/接口验证通过 | 按实际交付版本复核本说明 |
| C | 本机未找到C最终训练/预测代码，无法代为确认 | 确认训练、预测入口、阈值标签及实际关联均使用同一口径 |

最终提交前，A/B/C仍需共同确认上述时间口径，并核对C实际训练与预测实现。


# 26 个默认数值特征定义（B_v2）

令 R(t) 为末端 t 所对应五分钟区间 (t−5min,t] 的雨量。以下区间解释采用项目 L1/D1 约定；未被独立证明为官方时间语义。原始 event_ts 保留；预测键使用对齐后的 slot_end_epoch_s。所有历史精确位置以 epoch 秒匹配，缺槽不补零。

下表按 B_feature_api.NUMERIC_FEATURES 的顺序排列。station_id 是 C 编码的额外类别输入，threshold_mm 是 C 加入的额外数值输入；均不包含在这26列中。

| 名称 | 含义/公式 | 单位 | 历史范围（末端与区间） | 包含当前槽 | 缺失与特殊值处理 |
|---|---|---|---|---|---|
| current_5m_mm | 当前已完成五分钟区间的雨量 | mm | 末端 t；区间 (t−5min,t] | 是 | 仅保留有效观测锚点；当前槽缺失时不能产生该锚点 |
| lag_5m_mm | 末端为 t−5min 的单槽雨量 | mm | 精确末端 t−5min；对应其前五分钟区间 | 否 | 精确位置无有效记录则 null；不能取前一行替代 |
| rain_sum_15m_mm | 该窗口各完整五分钟区间累计雨量 | mm | t−10min,t−5min,t；区间 (t−15min,t] | 是 | 分别需要 3/6/12/3 个精确有效槽；有缺槽则 null，不作部分求和 |
| rain_sum_30m_mm | 该窗口各完整五分钟区间累计雨量 | mm | t−25min,…,t；区间 (t−30min,t] | 是 | 分别需要 3/6/12/3 个精确有效槽；有缺槽则 null，不作部分求和 |
| rain_sum_60m_mm | 该窗口各完整五分钟区间累计雨量 | mm | t−55min,…,t；区间 (t−60min,t] | 是 | 分别需要 3/6/12/3 个精确有效槽；有缺槽则 null，不作部分求和 |
| rain_sum_previous_15m_mm | 该窗口各完整五分钟区间累计雨量 | mm | t−25min,t−20min,t−15min；区间 (t−30min,t−15min] | 否 | 分别需要 3/6/12/3 个精确有效槽；有缺槽则 null，不作部分求和 |
| rain_change_15m_mm | 最近15分钟累计减前一段15分钟累计；表示两段变化，非物理加速度 | mm | t−25min,…,t；两段相邻15分钟窗口 | 是 | 任一累计为 null 则 null |
| rain_max_60m_mm | 最近60分钟单槽最大雨量 | mm | t−55min,…,t；12个精确末端 | 是 | 全部12槽有效才计算，否则 null |
| rain_std_60m_mm | 最近60分钟12个槽雨量的总体标准差（除以12） | mm | t−55min,…,t；12个精确末端 | 是 | 全部12槽有效才计算，否则 null |
| wet_slots_60m | 最近60分钟雨量 >0 的槽数，0–12 | 槽 | t−55min,…,t；12个精确末端 | 是 | 全部12槽有效才计算，否则 null |
| hour_sin | sin(2πh/24)；h=新加坡时刻的小时+分钟/60，m=月份1–12 | 无量纲 | 仅预测时刻 t 的本地日历；不读取雨量 | 不适用（只用时间） | 合法锚点时间可计算；不因历史雨量缺槽而置 null |
| hour_cos | cos(2πh/24)；h=新加坡时刻的小时+分钟/60，m=月份1–12 | 无量纲 | 仅预测时刻 t 的本地日历；不读取雨量 | 不适用（只用时间） | 合法锚点时间可计算；不因历史雨量缺槽而置 null |
| month_sin | sin(2π(m−1)/12)；h=新加坡时刻的小时+分钟/60，m=月份1–12 | 无量纲 | 仅预测时刻 t 的本地日历；不读取雨量 | 不适用（只用时间） | 合法锚点时间可计算；不因历史雨量缺槽而置 null |
| month_cos | cos(2π(m−1)/12)；h=新加坡时刻的小时+分钟/60，m=月份1–12 | 无量纲 | 仅预测时刻 t 的本地日历；不读取雨量 | 不适用（只用时间） | 合法锚点时间可计算；不因历史雨量缺槽而置 null |
| lag_10m_mm | 末端为 t−10min 的单槽雨量 | mm | 精确末端 t−10min；对应其前五分钟区间 | 否 | 精确位置缺失则 null |
| lag_15m_mm | 末端为 t−15min 的单槽雨量 | mm | 精确末端 t−15min；对应其前五分钟区间 | 否 | 精确位置缺失则 null |
| lag_30m_mm | 末端为 t−30min 的单槽雨量 | mm | 精确末端 t−30min；对应其前五分钟区间 | 否 | 精确位置缺失则 null |
| wet_current | current_5m_mm >0 为1，否则0 | 0/1 | 末端 t 的当前槽 | 是 | 有效当前锚点可计算；真实零为0 |
| wet_lag_5m | lag_5m_mm >0 为1，否则0 | 0/1 | 精确末端 t−5min | 否 | lag缺失则 null，不能将缺失判为无雨 |
| wet_lag_10m | lag_10m_mm >0 为1，否则0 | 0/1 | 精确末端 t−10min | 否 | lag缺失则 null，不能将缺失判为无雨 |
| consecutive_wet_slots_60m | 从当前槽向过去数连续正雨量槽；当前干则0；最多12 | 槽 | t−55min,…,t | 是 | 12槽不完整则 null；全湿记12并配左截断标记；槽内非持续降雨证明 |
| wet_run_left_censored_60m | 全部12槽均湿为1，表示雨段起点可能在窗口之前；否则0 | 0/1 | t−55min,…,t | 是 | 12槽不完整则 null |
| minutes_since_last_rain_60m_capped | t减最近有雨槽的末端时间（分钟）；未见湿槽时编码60 | 分钟/截断编码 | t−55min,…,t；观测到的间隔为0,5,…,55 | 是 | 12槽不完整则 null；完整但全干时60仅为未知间隔编码，必须配 no_rain_observed_60m |
| no_rain_observed_60m | 12个有效槽均干为1；窗口内有湿槽为0 | 0/1 | t−55min,…,t | 是 | 12槽不完整则 null；1不代表历史上从未下雨 |
| rain_change_5m_mm | current_5m_mm − lag_5m_mm | mm | 精确末端 t−5min 和 t | 是 | lag5缺失则 null |
| rain_ratio_15m_30m | 最近15分钟累计 / 最近30分钟累计 | 无量纲，0–1 | t−25min,…,t | 是 | 完整6槽且分母>0才相除；完整且分母0则置0（非负雨量故分子也0）；有缺槽则 null |

## 非默认模型输入的审计字段

- `n_valid_15m/n_valid_30m/n_valid_60m/n_valid_previous_15m`：有效槽数；`history_complete_60m`：12槽是否齐全。
- `minutes_since_last_rain_60m`：未截断编码的原始降雨距今时间，窗口内无湿槽时为 null；默认26列不包含它。
- `prediction_epoch_s/prediction_ts/year`：主键、日历、分区字段，不可整表直接作为模型数值输入。
- 连续湿槽数量乘5可描述已观测的连续湿槽覆盖分钟数，但不能断言槽内每秒都下雨；全湿时只能说明观察窗口内连续湿，不能推断雨段真实起点。
- 完整历史模型的推理入口要求12槽完整。此规则只看过去；不能用未来标签是否有效决定是否允许事前预测。


## 补充：完整历史筛选的逐年保留率

B保留不完整历史的锚点并保持null；完整历史模型训练/评价排除这些锚点。A的model_anchor_eligible同时要求未来和历史完整，因此本表不是最终训练保留率；两类损失不能直接相加。推理缺槽时接口拒绝预测。

In [14]:
show_csv(BROOT / 'results/features_v2/history_retention_by_year.csv', 8)


year,anchors,history_complete,history_incomplete,retention_rate,excluded_rate
2017,5256020,4941980,314040,0.9402513689065111,0.059748631093488935
2018,4594246,4028323,565923,0.8768191777279667,0.12318082227203331
2019,5026862,4965109,61753,0.9877153977968761,0.012284602203123884
2020,6334473,6264593,69880,0.9889683009146933,0.011031699085306657
2021,7044646,6995148,49498,0.9929736710687804,0.007026328931219572
2022,7133230,7091648,41582,0.9941706632198878,0.005829336780112215
2023,6742890,6704161,38729,0.9942563203611507,0.005743679638849253
2024,6406260,6356871,49389,0.9922905095953021,0.007709490404697927


显示 8 / 8 行；来源：/Users/zhu/Downloads/5208/task2/DSA5208_Project2/B_delivery/results/features_v2/history_retention_by_year.csv


## 补充：真实基础表关联证据与C的待办

下面是A正式标签表与B特征表的实际核验，尚不是C最终组装训练集的核验。C需在阈值展开之前调用下方函数，失败时停止，不应先去重掩盖问题。

In [15]:
print((BROOT / 'results/join_audit/validation.json').read_text())
show_csv(BROOT / 'results/join_audit/yearly_join_audit.csv', 8)


{
  "status": "PASS",
  "scope": "published_A_targets_vs_B_features_pre_sampling_pre_threshold",
  "C_actual_training_input_status": "PENDING_NOT_AVAILABLE_LOCALLY",
  "totals": {
    "left_rows": 48538627,
    "right_rows": 48538627,
    "left_unique_keys": 48538627,
    "right_unique_keys": 48538627,
    "left_duplicate_key_groups": 0,
    "right_duplicate_key_groups": 0,
    "left_duplicate_excess_rows": 0,
    "right_duplicate_excess_rows": 0,
    "left_null_key_rows": 0,
    "right_null_key_rows": 0,
    "left_unmatched_rows": 0,
    "right_unmatched_rows": 0,
    "inner_join_rows": 48538627,
    "left_join_rows": 48538627,
    "row_inflation": 0,
    "left_match_rate": 1.0,
    "right_match_rate": 1.0,
    "status": "PASS"
  },
  "year_partition_consistency": true,
  "input_integrity": {
    "status": "PASS",
    "scope": "full",
    "checked_files": 549,
    "release_id": "rainfall_v1_r3",
    "data_version": "v1",
    "config_hash": "e5258b99925606c8ba88207cb44a6756514fa84cd60f

scope,left_unique_keys,left_rows,left_duplicate_key_groups,left_duplicate_excess_rows,left_null_key_rows,right_unique_keys,right_rows,right_duplicate_key_groups,right_duplicate_excess_rows,right_null_key_rows,left_unmatched_rows,right_unmatched_rows,matched_left_rows,inner_join_rows,left_join_rows,left_match_rate,right_match_rate,row_inflation,status,year
published_targets_vs_B_features_2017,5256020,5256020,0,0,0,5256020,5256020,0,0,0,0,0,5256020,5256020,5256020,1.0,1.0,0,PASS,2017
published_targets_vs_B_features_2018,4594246,4594246,0,0,0,4594246,4594246,0,0,0,0,0,4594246,4594246,4594246,1.0,1.0,0,PASS,2018
published_targets_vs_B_features_2019,5026862,5026862,0,0,0,5026862,5026862,0,0,0,0,0,5026862,5026862,5026862,1.0,1.0,0,PASS,2019
published_targets_vs_B_features_2020,6334473,6334473,0,0,0,6334473,6334473,0,0,0,0,0,6334473,6334473,6334473,1.0,1.0,0,PASS,2020
published_targets_vs_B_features_2021,7044646,7044646,0,0,0,7044646,7044646,0,0,0,0,0,7044646,7044646,7044646,1.0,1.0,0,PASS,2021
published_targets_vs_B_features_2022,7133230,7133230,0,0,0,7133230,7133230,0,0,0,0,0,7133230,7133230,7133230,1.0,1.0,0,PASS,2022
published_targets_vs_B_features_2023,6742890,6742890,0,0,0,6742890,6742890,0,0,0,0,0,6742890,6742890,6742890,1.0,1.0,0,PASS,2023
published_targets_vs_B_features_2024,6406260,6406260,0,0,0,6406260,6406260,0,0,0,0,0,6406260,6406260,6406260,1.0,1.0,0,PASS,2024


显示 8 / 8 行；来源：/Users/zhu/Downloads/5208/task2/DSA5208_Project2/B_delivery/results/join_audit/yearly_join_audit.csv


### 可编辑的一对一关联核验函数

左侧为C实际筛选/采样后的监督锚点，右侧为B特征；两侧主键都必须唯一，左侧应全部匹配。B全量右侧中未使用的锚点不一定是错误。阈值展开后需改用三列主键。

In [16]:
KEYS = ['station_id', 'prediction_epoch_s']

def audit_anchor_join(left,right,scope='C_pre_expansion',require_all_left=True,require_all_right=False):
    """left = C supervised anchors; right = B features. Invoke on actual C inputs.

    Audit precedes threshold expansion: repeated anchor/threshold rows are not valid inputs here.
    Returns evidence; caller performs the join only after PASS. No rain/target values are collected.
    """
    from pyspark.sql import functions as F
    def describe(frame):
        keys=frame.select(*KEYS)
        nulls=keys.filter(F.col(KEYS[0]).isNull()|F.col(KEYS[1]).isNull()).count()
        counts=keys.groupBy(*KEYS).count()
        info=counts.agg(F.count('*').alias('unique_keys'),F.sum('count').alias('rows'),
            F.sum(F.when(F.col('count')>1,F.lit(1)).otherwise(0)).alias('duplicate_key_groups'),
            F.sum(F.col('count')-1).alias('duplicate_excess_rows')).first().asDict()
        return keys,{k:int(v or 0) for k,v in info.items()}|{'null_key_rows':nulls}
    lk,ls=describe(left);rk,rs=describe(right)
    result={'scope':scope,**{'left_'+k:v for k,v in ls.items()},**{'right_'+k:v for k,v in rs.items()}}
    clean=ls['null_key_rows']==rs['null_key_rows']==ls['duplicate_excess_rows']==rs['duplicate_excess_rows']==0
    if not clean:
        result.update(status='FAIL',reason='Null or duplicate keys; join was not executed')
        return result
    unmatched_left=lk.join(rk,KEYS,'left_anti').count()
    unmatched_right=rk.join(lk,KEYS,'left_anti').count()
    inner_rows=lk.join(rk,KEYS,'inner').count()
    left_join_rows=lk.join(rk,KEYS,'left').count()
    result.update(left_unmatched_rows=unmatched_left,right_unmatched_rows=unmatched_right,
        matched_left_rows=ls['rows']-unmatched_left,inner_join_rows=inner_rows,left_join_rows=left_join_rows,
        left_match_rate=(ls['rows']-unmatched_left)/ls['rows'] if ls['rows'] else None,
        right_match_rate=(rs['rows']-unmatched_right)/rs['rows'] if rs['rows'] else None,
        row_inflation=left_join_rows-ls['rows'])
    ok=(inner_rows==ls['rows']-unmatched_left and left_join_rows==ls['rows']
        and (not require_all_left or unmatched_left==0) and (not require_all_right or unmatched_right==0))
    result['status']='PASS' if ok else 'FAIL'
    return result


### C实际训练输入中的调用示例（默认不运行）

C须把 C_ANCHORS 和 B_FEATURES 替换为其真实的展开前DataFrame；不能把当前示例当C已经验收。

In [17]:
RUN_C_ACTUAL_JOIN_AUDIT = False
if RUN_C_ACTUAL_JOIN_AUDIT:
    assert 'C_ANCHORS' in globals() and 'B_FEATURES' in globals(), '请先提供C真实展开前锚点和B特征DataFrame'
    check = audit_anchor_join(C_ANCHORS, B_FEATURES, scope='C_actual_pre_threshold_expansion')
    dump(BROOT / 'results/join_audit/C_actual_input_validation.json', check)
    assert check['status'] == 'PASS', check
else:
    print('C实际训练输入关联核验待C执行；基础表核验已经通过。')


C实际训练输入关联核验待C执行；基础表核验已经通过。


## 运行环境补充

硬件配置是本次补查，峰值内存没有测量。两类原运行时间含实际读取与写出，未包含后续报告/图表/哈希与Spark关闭；后补独立核验不计入旧耗时，详细起止点见中文报告。

In [18]:
print((BROOT / 'results/hardware_inventory.json').read_text())


{
  "chip_type": "Apple M5",
  "machine_name": "MacBook Air",
  "number_processors": "proc 10:4:6:0",
  "physical_memory": "24 GB",
  "checked_at": "2026-10-11T03:31:30.607086+00:00",
  "source": "system_profiler SPHardwareDataType; identifiers excluded",
  "measurement_scope": "hardware inventory obtained after the timed runs; peak memory not measured"
}

